<h1 style=\"text-align: center; font-size: 75px;\"> ⚙️ Run Workflow: Agentic Audio RAG </h1>

📘 Project Overview: 
 This notebook demonstrates a modular architecture for answering natural language questions 
 over one or more transcribed audio/video files documents using only local and open-source models (e.g., LLaMA.cpp, OpenAI Whisper).
 The system processes transcript documents chunk-by-chunk and synthesizes a final answer using a multi-step LLM workflow.

# Notebook Overview

- Start Execution
- Define User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- KV Memory
- LLM Setup
- State Model
- Node Functions
- Graph Definition
- Graph Visualization
- Generated Answer
- Message History

# Start Execution

In [ ]:
# Standard library imports
import os  # Provides OS-related utilities
import sys  # Allows manipulation of Python runtime environment
import time  # Enables time-based operations
from pathlib import Path  # Object-oriented file system paths

# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

# Local application-specific imports
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state
from src.model_selection import ModelSelector
from src.utils import (  # Utility functions for logging, LLM I/O, and schema generation
    load_config,
    load_secrets,
    load_secrets_to_env,
    configure_proxy,
    display_image,
    get_project_root,
    get_response_from_llm,
    json_schema_from_type,
    log_timing,
    sec_to_timestamp,
    logger,
    login_huggingface,
    setup_model_environment,
    ensure_wav,
    initialize_llm
)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

# Define User Constants

In [ ]:
QUESTION: str = "What are the main issues regarding the product"
DOCS: list
FILE_ID: str
MEMORY: SimpleKVMemory
INPUT_PATH: Path = Path("../data/input")

# Install and Import Libraries

In [4]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 11.6 ms, sys: 23.4 ms, total: 35 ms
Wall time: 1.69 s


In [ ]:
from __future__ import annotations  # Enables postponed evaluation of annotations (PEP 563)

# ─────── Standard Library ───────
import base64  # Provides encoding and decoding of binary data
import functools  # Higher-order functions for functional programming
import json  # JSON serialization and deserialization
import logging  # Flexible logging system
import multiprocessing  # Support for spawning processes
import shutil  # High-level file operations
import warnings  # Issue warning messages
from collections import namedtuple  # Factory for creating tuple subclasses with named fields
from datetime import datetime  # Date and time utilities
from pathlib import Path  # Object-oriented filesystem paths
from typing import Any, Dict, List, Tuple, Literal, Optional, TypedDict  # Type hinting support
import numpy as np  # Numerical operations and array handling

# ─────── Third-Party Packages ───────
import yaml  # YAML parsing and serialization
from IPython.display import Markdown, display  # IPython utilities for notebook output formatting
from tqdm import tqdm  # Visual progress bar for loops
import torch
from huggingface_hub import snapshot_download, hf_hub_download
import soundfile as sf  # Library for reading and writing sound files

# Qwen Omni (audio+video+text) – both full & Thinker-only variants
from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniThinkerForConditionalGeneration
from transformers import AutoProcessor as ClapProcessor, ClapModel
from qwen_omni_utils import process_mm_info     # official utils to prep audio/video inputs

# ─────── LangChain Core & Community ───────
from langchain.docstore.document import Document  # Core document abstraction
from langchain_community.llms import LlamaCpp  # Integration for local LlamaCpp models

from src.agentic_workflow import build_agentic_graph
from src.agentic_audio_rag_model import AgenticAudioRAGModel  # Custom model for audio RAG tasks
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state

# Configure Settings

In [6]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [ ]:
project_root = get_project_root()
MEMORY_PATH: Path = Path("../data/memory")
INPUT_PATH: Path = Path("../data/input")
CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"

# Default model path in case Audio Models fail to download
LLAMA_MODEL_PATH = "/home/jovyan/datafabric/meta-llama3.1-8b-Q8/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"
SAMPLE_MEDIA_PATH = INPUT_PATH / "sample_tts.mp3"

CONTEXT_WINDOW = 8192
MAX_TOKENS = CONTEXT_WINDOW // 8
CHUNK_SIZE = CONTEXT_WINDOW // 2
CHUNK_OVERLAP = CHUNK_SIZE // 16

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Configuration and Secrets Loading

In this section, we load configuration parameters and API keys from separate YAML files. This separation helps maintain security by keeping sensitive information (API keys) separate from configuration settings.

- **config.yaml**: Contains non-sensitive configuration parameters like model sources and URLs
- **secrets.yaml**: Contains sensitive API keys for services like HuggingFace
- *(Optional for Premium users)* Secrets such as API keys for services like HuggingFace can be stored as environment variables for the project and loaded into the notebook (see the project's README file for steps on how to save secrets in Secrets Manager).

In [ ]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

In [ ]:
configure_proxy(config)

## Create a Sample Audio File

In [ ]:
# login_huggingface(secrets)

# wav = hf_hub_download(
#         repo_id="roneneldan/TEDLIUM_sample",
#         filename="TED_0001.wav",
#         cache_dir=INPUT_PATH,
# )
# print("Saved to:", wav)

import pyttsx3

if not SAMPLE_MEDIA_PATH.exists():
    tts = pyttsx3.init()                 # offline, uses system voices
    tts.setProperty("rate", 120)         # speaking speed
    text = (
        "Hello and welcome to the Agentic Audio RAG demo."
        "This short clip will be chunked and analysed by the workflow."
        "Feel free to ask any question about its content."
    )
    tts.save_to_file(text, str(SAMPLE_MEDIA_PATH))
    tts.runAndWait()
    tts.stop()
    print("🎙️  Generated synthetic audio →", SAMPLE_MEDIA_PATH)
else:
    print("🎙️  Using existing file →", SAMPLE_MEDIA_PATH)

# Verify Assets

In [ ]:
def log_asset_status(asset_path: str, asset_name: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured.")
    else:
        logger.info(f"{asset_name} is not properly configured. Please ensure the required asset is correctly configured in your AI Studio project according to the README file.")

def log_secrets_status(secrets: Dict[str, Any], success_message: str, failure_message: str) -> None:
    """
    Logs the status of secrets based on their existence.

    Parameters:
        secrets (Dict[str, Any]): Secrets retrieved to check if they exist.
        success_message (str): Message to log if secrets exists.
        failure_message (str): Message to log if secrets do not exist.
    """
    if secrets:
        logger.info(f"Project secrets are available. {success_message}")
    else:
        logger.info(f"There are no project secrets found. {failure_message}")

In [ ]:
log_asset_status(
    asset_path=INPUT_PATH,
    asset_name="Input Data",
)
log_asset_status(
    asset_path=LLAMA_MODEL_PATH,
    asset_name="LLM",
)

log_asset_status(
    asset_path=CONFIG_PATH,
    asset_name="Config",
    success_message="",
    failure_message="Please check if the configs.yaml was propely connfigured in your project on AI Studio."
)

log_secrets_status(
    secrets=secrets,
    success_message="",
    failure_message="Please check if the secrets were propely connfigured in your secrets yaml file or in Secrets Manager."
)

In [ ]:
assert SAMPLE_MEDIA_PATH.exists(), "Sample media file not found. Please ensure the required asset is correctly configured in your AI Studio project according to the README file."

# Setup Model

In [ ]:
# Login to Hugging Face (required for downloading gated models)
try:
    login_huggingface(secrets)
    logger.info("✅ Hugging Face authentication successful")
except Exception as e:
    logger.warning(f"⚠️ Hugging Face authentication failed: {e}")
    logger.info("Some models may not be accessible if they require authentication")

In [ ]:
logger.info("Scanning diretory for media files: %s", INPUT_PATH)

AUDIO_LLM = "Qwen/Qwen2.5-Omni-7B"               # audio/video multimodal, reasoning + (optional) speech out
CLAP_ID = "laion/clap-htsat-unfused"             # text↔audio embedding for retrieval

# Supported media types (audio + video)
AUDIO_EXTS = {".mp3", ".wav", ".ogg", ".flac", ".m4a"}
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}
MEDIA_EXTS = AUDIO_EXTS | VIDEO_EXTS

# Make HF cache live inside project (per README & utils)
setup_model_environment()  # keep your project-local HF cache layout

In [ ]:
selector = ModelSelector()
local_model_dir = Path(selector.format_model_path(AUDIO_LLM))
local_model_dir.mkdir(parents=True, exist_ok=True)

# Qwen
local_llm_dir = Path(selector.format_model_path(AUDIO_LLM)); 
local_llm_dir.mkdir(parents=True, exist_ok=True)
if not any(local_llm_dir.iterdir()):
    logger.info("⬇️ Downloading Audio LLM model %s → %s", AUDIO_LLM, str(local_llm_dir))
    snapshot_download(
        repo_id=AUDIO_LLM, 
        local_dir=str(local_llm_dir), 
        local_dir_use_symlinks=False, 
        resume_download=True,
        token=os.environ.get("AIS_HUGGINGFACE_API_KEY")
    )
AUDIO_LLM_MODEL_PATH = str(local_llm_dir)


# CLAP
local_clap_dir = Path(selector.format_model_path(CLAP_ID)); 
local_clap_dir.mkdir(parents=True, exist_ok=True)
if not any(local_clap_dir.iterdir()):
    logger.info("⬇️ Downloading the CLAP model %s → %s", CLAP_ID, str(local_clap_dir))
    snapshot_download(
        CLAP_ID, 
        local_dir=str(local_clap_dir), 
        local_dir_use_symlinks=False, 
        resume_download=True,
        token=os.environ.get("AIS_HUGGINGFACE_API_KEY")
    )
CLAP_LOCAL_DIR = str(local_clap_dir)

logger.info("🎙️ LLM model directory: %s", AUDIO_LLM_MODEL_PATH)
logger.info("🎵 CLAP model directory: %s", CLAP_LOCAL_DIR)

In [ ]:
# Qwen processor + text-only (Thinker) head for answers
processor = Qwen2_5OmniProcessor.from_pretrained(AUDIO_LLM_MODEL_PATH)
model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    AUDIO_LLM_MODEL_PATH, torch_dtype="auto", device_map="auto"
)

# CLAP: joint text/audio embedding model
clap_processor = ClapProcessor.from_pretrained(CLAP_LOCAL_DIR)
clap_model = ClapModel.from_pretrained(CLAP_LOCAL_DIR)
clap_model.eval()
clap_device = "cuda" if torch.cuda.is_available() else "cpu"
clap_model.to(clap_device)


In [ ]:
def segment_audio(wav_path: str, window_s: float = 30.0, hop_s: float = 15.0) -> List[Tuple[int, int, np.ndarray, int]]:
    """
    Return a list of segments as (start_sample, end_sample, waveform[np.float32 mono], sr).
    """
    audio, sr = sf.read(wav_path)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    n = len(audio); win = int(window_s * sr); hop = int(hop_s * sr)
    if n == 0:
        return []
    segs = []
    i = 0
    while i < n:
        j = min(i + win, n)
        segs.append((i, j, audio[i:j].astype(np.float32), sr))
        if j == n:
            break
        i += hop
    return segs

def clap_embed_audio(wav: np.ndarray, sr: int) -> np.ndarray:
    with torch.no_grad():
        inp = clap_processor(audios=wav, sampling_rate=sr, return_tensors="pt").to(clap_device)
        out = clap_model.get_audio_features(**inp)
        vec = out.cpu().numpy()[0]
        vec = vec / (np.linalg.norm(vec) + 1e-12)
        return vec.astype(np.float32)

def clap_embed_text(query: str) -> np.ndarray:
    with torch.no_grad():
        inp = clap_processor(text=query, return_tensors="pt").to(clap_device)
        out = clap_model.get_text_features(**inp)
        vec = out.cpu().numpy()[0]
        vec = vec / (np.linalg.norm(vec) + 1e-12)
        return vec.astype(np.float32)

class AudioIndex:
    def __init__(self, dim: int = 512):
        self.index = faiss.IndexFlatIP(dim)
        self.meta: List[Dict[str, Any]] = []

    def add(self, vecs: np.ndarray, metas: List[Dict[str, Any]]):
        self.index.add(vecs)
        self.meta.extend(metas)

    def search(self, qvec: np.ndarray, k: int = 8) -> List[Dict[str, Any]]:
        D, I = self.index.search(qvec[np.newaxis, :], k)
        out = []
        for idx, score in zip(I[0], D[0]):
            if 0 <= idx < len(self.meta):
                m = dict(self.meta[idx]); m["score"] = float(score)
                out.append(m)
        return out

# Build index over INPUT_PATH
logger.info("📂 Scanning media: %s", INPUT_PATH)
audio_index = AudioIndex(dim=512)  # CLAP default proj dim
docs_for_ui: List[Document] = []

media_paths = []
for p in sorted(Path(INPUT_PATH).rglob("*")):
    if any(part.startswith(".") and part not in {".", ".."} for part in p.parts):
        continue
    if p.is_file() and p.suffix.lower() in MEDIA_EXTS:
        media_paths.append(p)

for media_path in media_paths:
    # normalize to wav (extract audio from video if needed)
    wav = ensure_wav(str(media_path))
    segs = segment_audio(wav, window_s=30.0, hop_s=15.0)

    # embed each segment
    vecs, metas = [], []
    for (s0, s1, wav_seg, sr) in segs:
        v = clap_embed_audio(wav_seg, sr); vecs.append(v)
        start_s, end_s = s0 / sr, s1 / sr
        metas.append({
            "file_path": str(media_path),
            "file_name": media_path.name,
            "start_s": float(start_s),
            "end_s": float(end_s),
            "wav_path": wav,    # original wav path (we can also write segment files if you prefer)
        })

    if vecs:
        audio_index.add(np.stack(vecs, axis=0), metas)

    # For UI/debug: keep a Document shell with coarse timestamp span
    if segs:
        duration_s = segs[-1][1] / segs[-1][3]
        docs_for_ui.append(Document(
            page_content=f"[Audio] {media_path.name} ({duration_s:.1f}s)",
            metadata={
                "file_path": str(media_path),
                "file_name": media_path.name,
                "media_type": "audio" if media_path.suffix.lower() in AUDIO_EXTS else "video",
                "segments": [{"start": 0.0, "end": float(duration_s), "text": ""}],
                "source": "clap-index",
            },
        ))

logger.info("📇 Indexed %d media files, %d segments", len(media_paths), len(audio_index.meta))


In [ ]:
def retrieve_audio_segments(query: str, top_k: int = 6) -> List[Dict[str, Any]]:
    qvec = clap_embed_text(query)
    return audio_index.search(qvec, k=top_k)

# quick smoke test
# hits = retrieve_audio_segments("action items and deadlines", top_k=4); hits[:2]


In [ ]:
def qwen_answer_with_audio(question: str, hits: List[Dict[str, Any]], return_audio: bool = False) -> Dict[str, Any]:
    """
    Feed the top-K retrieved audio segments (CLAP) into Qwen Omni and ask it to answer.
    No explicit ASR step; Qwen perceives + reasons over the audio directly.
    """
    # Build a conversation: system grounding + user provides N audio clips + the question
    # We’ll attach each segment as an "audio" item and prepend a short text instruction.
    user_content = [{"type": "text", "text": f"Question: {question}"}]
    for h in hits:
        # slice the specific segment to a temp array (so Qwen only 'hears' the relevant window)
        audio_full, sr = sf.read(h["wav_path"])
        if audio_full.ndim == 2:
            audio_full = audio_full.mean(axis=1)
        s0, s1 = int(h["start_s"] * sr), int(h["end_s"] * sr)
        # guard
        s0 = max(0, min(s0, len(audio_full)))
        s1 = max(0, min(s1, len(audio_full)))
        seg = audio_full[s0:s1].astype(np.float32)
        # attach as an in-memory audio clip (qwen_omni_utils handles proper packaging)
        user_content.append({"type": "audio", "audio": seg, "sampling_rate": sr})

    conversation = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": "You are a helpful analyst. Carefully listen to the audio clips and answer using only their content. Quote timestamps if useful."}
            ],
        },
        {"role": "user", "content": user_content},
    ]

    # 1) Build the text part using the chat template
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)

    # 2) Build the multimodal tensors (audios/images/videos) from the same conversation
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)

    # 3) Final packed inputs for the model
    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=False,
    ).to(model.device).to(model.dtype)

    # 4) Generate
    with torch.no_grad():
        if return_audio:
            text_ids, audio = model.generate(**inputs, use_audio_in_video=False)
        else:
            text_ids = model.generate(**inputs, use_audio_in_video=False)

    answer = processor.batch_decode(
        text_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()

    return {
        "answer": answer,
        "evidence": [
            {
                "file_name": h["file_name"],
                "file_path": h["file_path"],
                "start_s": h["start_s"],
                "end_s": h["end_s"],
                "score": h["score"],
            }
            for h in hits
        ],
        # "audio": audio if return_audio else None,
    }


In [ ]:
# === Qwen adapter used by the agentic graph ===

class QwenOmniAgent:
    """Minimal adapter so the graph can call `llm.answer(question, hits)`."""
    def __init__(self, processor, model):
        self.processor = processor
        self.model = model
        self.device = getattr(model, "device", "cuda" if torch.cuda.is_available() else "cpu")

    def answer(self, question: str, audio_hits: list, return_audio: bool = False) -> dict:
        """
        audio_hits: list of dicts with keys: file_path, file_name, start_s, end_s, score, wav_path
        returns: {"answer": str, "evidence": [...]} (+ "audio" if return_audio=True and full Omni is used)
        """
        import numpy as np, soundfile as sf
        from qwen_omni_utils import process_mm_info

        # Build a system+user conversation where user attaches the audio segments
        user_content = [{"type": "text", "text": f"Question: {question}"}]
        # Attach each audio segment in-memory (so Qwen 'hears' only the relevant windows)
        for h in audio_hits:
            audio_full, sr = sf.read(h["wav_path"])
            if audio_full.ndim == 2:
                audio_full = audio_full.mean(axis=1)
            s0, s1 = int(h["start_s"] * sr), int(h["end_s"] * sr)
            s0 = max(0, min(s0, len(audio_full)))
            s1 = max(0, min(s1, len(audio_full)))
            seg = audio_full[s0:s1].astype(np.float32)
            user_content.append({"type": "audio", "audio": seg, "sampling_rate": sr})

        conversation = [
            {"role": "system", "content": [{"type": "text", "text": "You are a precise analyst. Listen to the audio clips and answer using only their content. Keep timestamps where helpful."}]},
            {"role": "user",   "content": user_content},
        ]

        # 1) Create the text prompt from the chat template
        text = self.processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)

        # 2) Collect multimodal blobs (audio) from the same conversation
        audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)

        # 3) Pack tensors for the model
        inputs = self.processor(
            text=text, audio=audios, images=images, videos=videos,
            return_tensors="pt", padding=True, use_audio_in_video=False
        ).to(self.model.device).to(self.model.dtype)

        # 4) Generate
        with torch.no_grad():
            if return_audio:
                text_ids, audio = self.model.generate(**inputs, use_audio_in_video=False)
            else:
                text_ids = self.model.generate(**inputs, use_audio_in_video=False)

        answer = self.processor.batch_decode(
            text_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

        out = {
            "answer": answer,
            "evidence": [
                {
                    "file_name": h["file_name"],
                    "file_path": h["file_path"],
                    "start_s": h["start_s"],
                    "end_s": h["end_s"],
                    "score": h["score"],
                }
                for h in audio_hits
            ],
        }
        # if return_audio: out["audio"] = audio
        return out

# Instantiate once and put in notebook scope
qwen_agent = QwenOmniAgent(omni_processor, omni_model)


In [ ]:
# def transcribe_with_seamless_m4t(media_path: str, model_dir: str) -> Dict[str, Any]:
#     """
#     Transcribe an audio/video file using Meta's SeamlessM4T-v2 (US-hosted).
#     Returns:
#       {"text": <full transcript>, "segments": [{"start": float, "end": float, "text": str}, ...]}
#     """
#     device = "cuda" if torch.cuda.is_available() else "cpu"

#     # Load processor + model once per cell execution (fast enough for notebook)
#     processor = AutoProcessor.from_pretrained(model_dir)
#     model = SeamlessM4Tv2Model.from_pretrained(model_dir).to(device)

#     # Normalize media → 16kHz mono wav (works for audio OR video)
#     wav_path = ensure_wav(media_path)

#     # Raw audio loading
#     import soundfile as sf
#     audio, sr = sf.read(wav_path)
#     if audio.ndim == 2:  # mono
#         audio = audio.mean(axis=1)

#     # Prepare inputs and generate TEXT (no speech synthesis)
#     inputs = processor(audios=audio, sampling_rate=sr, return_tensors="pt").to(device)
#     # english target text; change "eng" if you need another iso code
#     outputs = model.generate(**inputs, tgt_lang="eng", generate_speech=False)

#     # Seamless v2 returns token IDs → decode to text
#     text = processor.decode(outputs[0].tolist()[0], skip_special_tokens=True).strip()

#     # If timestamp segmentation isn’t available, create a coarse span
#     est_seconds = max(10.0, len(text.split()) / 2.5)
#     segments = [{"start": 0.0, "end": float(est_seconds), "text": text}]

#     return {"text": text, "segments": segments}

In [ ]:
# from huggingface_hub import list_repo_files
# print([f for f in list_repo_files("mistralai/Mistral-7B-Instruct-v0.3")])

# Whisper Model and Transcript Setup

In [ ]:
# %%time

# import whisper

# whisper_model = whisper.load_model("large-v3")
# raw_transcript = whisper_model.transcribe(str(SAMPLE_MEDIA_PATH), fp16=False)
# full_transcript = raw_transcript["text"].strip()


In [ ]:
# FILE_ID = SAMPLE_MEDIA_PATH.name
# DOCS = [Document(page_content=full_transcript)]
# print(full_transcript[:300] + "...")

# LLM Setup

In [ ]:
# %%time

# if not os.path.exists(gguf_path):
#     model_path = str(MODEL_PATH)
# else:
#     model_path = str(gguf_path)

# llm = LlamaCpp(
#     model_path=model_path,
#     n_gpu_layers=-1,                             
#     n_batch=512,                                 
#     n_ctx=CONTEXT_WINDOW,
#     max_tokens=MAX_TOKENS,
#     f16_kv=True,
#     use_mmap=False,                             
#     low_vram=False,                            
#     rope_scaling=None,
#     temperature=0.0,
#     repeat_penalty=1.0,
#     streaming=False,
#     stop=None,
#     seed=42,
#     num_threads=multiprocessing.cpu_count(),
#     verbose=False                                
# )

CPU times: user 1.18 s, sys: 2.68 s, total: 3.86 s
Wall time: 1min 13s


# KV Memory

In [ ]:
MEMORY = SimpleKVMemory(MEMORY_PATH)

# Build and Compile LangGraph

In [ ]:
# === Agentic Audio RAG graph (LangGraph) ===
try:
    from langgraph.graph import StateGraph, END
except Exception as e:
    raise RuntimeError("LangGraph not installed. Please `pip install langgraph`.") from e

# ---- State shape ----
# state = {
#   "question": str,
#   "index": AudioIndex,
#   "docs": List[Document] (optional, for UI),
#   "llm": QwenOmniAgent,
#   "memory": dict (optional),
#   "hits": List[dict] (retrieved audio segments),
#   "answer": str,
#   "evidence": List[dict],
# }

def node_check_memory(state: dict) -> dict:
    q = state["question"].strip()
    mem = state.get("memory") or {}
    if q in mem:
        cached = mem[q]
        state["answer"] = cached["answer"]
        state["evidence"] = cached.get("evidence", [])
        state["from_cache"] = True
    else:
        state["from_cache"] = False
    return state

def node_retrieve(state: dict) -> dict:
    if state.get("from_cache"):
        return state
    # text -> audio retrieval via CLAP
    state["hits"] = retrieve_audio_segments(state["question"], top_k=6)
    return state

def node_generate(state: dict) -> dict:
    if state.get("from_cache"):
        return state
    # ask Qwen directly with the audio clips
    out = state["llm"].answer(state["question"], state["hits"], return_audio=False)
    state["answer"] = out["answer"]
    state["evidence"] = out["evidence"]
    return state

def node_update_memory(state: dict) -> dict:
    mem = state.get("memory")
    if isinstance(mem, dict) and not state.get("from_cache"):
        mem[state["question"]] = {"answer": state["answer"], "evidence": state.get("evidence", [])}
    return state

def build_audio_agent_graph():
    sg = StateGraph(dict)
    sg.add_node("memory",   node_check_memory)
    sg.add_node("retrieve", node_retrieve)
    sg.add_node("generate", node_generate)
    sg.add_node("memoize",  node_update_memory)

    sg.set_entry_point("memory")
    sg.add_edge("memory",   "retrieve")
    sg.add_edge("retrieve", "generate")
    sg.add_edge("generate", "memoize")
    sg.add_edge("memoize",  END)
    return sg.compile()


In [ ]:
graph = build_agentic_graph()
compiled_graph = graph.compile()

In [ ]:
png = compiled_graph.get_graph().draw_mermaid_png()
display_image(png)

# Run Agentic Workflow

In [ ]:
%%time

final_graph = compiled_graph.invoke(
        input={
            "docs": DOCS,
            "file_id": FILE_ID,
            "question": QUESTION,
            "input_path": INPUT_PATH, 
            "memory": MEMORY, 
            "llm": llm,
            "messages": [],
        },
    )

# Generated Answer and Snippets

In [ ]:
answer = final_graph.get('answer')
display(Markdown(answer))

In [ ]:
for snippet in final_graph.get('snippets'):
    display(Markdown(snippet))

# Message History

In [ ]:
pretty_json = json.dumps(final_graph.get('messages'), indent=4)
print(pretty_json)

In [20]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).